# Project 4, Sample code notebook
## Yoga in the second trimester and gestational diabetes

Fictional, AI-simulated data, for teaching structure only, not medical advice.

This is a worked sample notebook for [Project 4](project-4). It is a two-group comparison: the binary explanatory variable is yoga (yes/no) and the binary response is maternal (gestational) diabetes (yes/no). Because this is an *observational* design, we also record a likely confounder, SES, and look at the comparison *within* SES levels (subclassification).

Your own project must use your instructor-assigned question and stay limited to a single two-group comparison, do not add a second explanatory contrast.

Workflow: simulate → describe (proportions + plot) → check the confounder (faceted) → test (by chance).

In [ ]:
library(tidyverse)

## 1, Simulate the data

This follows the binary outcome, confounding template (Case 6) from [Demo lab 1]: a confounder (SES) drives both the explanatory variable (yoga) and the binary response (maternal diabetes), and yoga has no true effect. `set.seed()` makes the random draw reproducible, so any crude yoga, diabetes association is produced by SES confounding plus chance.

In [ ]:
set.seed(109)
n <- 200   # enough for expected counts >= 5 in a 2x2 table

# Confounder: SES (about 40% high, 60% low)
SES <- sample(c("high", "low"), size = n, replace = TRUE, prob = c(0.4, 0.6))

# Yoga participation depends on SES (high-SES people more likely to attend)
p_yoga <- ifelse(SES == "high", 0.70, 0.20)
yoga <- ifelse(rbinom(n, size = 1, prob = p_yoga) == 1, "yes", "no")

# Maternal diabetes depends ONLY on SES -- true causal effect of yoga is zero
p_diabetes <- ifelse(SES == "high", 0.10, 0.30)
maternal_diabetes <- ifelse(rbinom(n, size = 1, prob = p_diabetes) == 1, "yes", "no")

# One row per person; columns for the three variables
study_data <- tibble(
  SES = factor(SES, levels = c("low", "high")),
  yoga = factor(yoga, levels = c("no", "yes")),
  maternal_diabetes = factor(maternal_diabetes, levels = c("no", "yes"))
)

glimpse(study_data)
head(study_data)

## 2, Describe: counts and proportions by group

Two categorical variables → a two-way count and the proportion with diabetes within each yoga group.

In [ ]:
# Crude 2x2 counts
study_data |>
  count(yoga, maternal_diabetes)

# Diabetes rate within each yoga group
study_data |>
  group_by(yoga) |>
  summarize(n = n(),
            diabetes = sum(maternal_diabetes == "yes"),
            prop_diabetes = mean(maternal_diabetes == "yes"))

## 3, Describe: 100% stacked bar chart (crude)

In [ ]:
study_data |>
  count(yoga, maternal_diabetes) |>
  ggplot(aes(x = yoga, y = n, fill = maternal_diabetes)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = scales::percent_format()) +
  labs(title = "Maternal diabetes by yoga group (100% stacked)",
       x = "Yoga participation", y = "Percent within yoga group",
       fill = "Maternal diabetes")

## 4, Check the confounder (subclassification + faceted plot)

Because the design is observational, look at the confounder before any test. Compare yoga groups within SES levels (subclassification) and in a faceted 100% stacked bar. If the crude difference mostly disappears inside each SES stratum, that points to confounding by SES, not a causal effect of yoga.

In [ ]:
# Diabetes rate by yoga group, within each SES level
study_data |>
  group_by(SES, yoga) |>
  summarize(n = n(),
            prop_diabetes = mean(maternal_diabetes == "yes"),
            .groups = "drop")

# Faceted 100% stacked bars by SES
study_data |>
  count(SES, yoga, maternal_diabetes) |>
  ggplot(aes(x = yoga, y = n, fill = maternal_diabetes)) +
  geom_col(position = "fill") +
  facet_wrap(~ SES) +
  scale_y_continuous(labels = scales::percent_format()) +
  labs(title = "Maternal diabetes by yoga group, within SES strata",
       x = "Yoga participation", y = "Percent within yoga group",
       fill = "Maternal diabetes")

## 5, Test the "by chance" explanation

Having seen that SES is imbalanced across yoga groups, now address the remaining explanation, chance. Binary explanatory × binary response → `prop.test()` for the difference in diabetes proportions (with a 95% CI), and `chisq.test()` on the 2×2 table. Check that expected counts are ≥ 5. A small p-value speaks only to chance; it cannot undo the confounding you already saw.

In [ ]:
by_arm <- study_data |>
  group_by(yoga) |>
  summarize(x = sum(maternal_diabetes == "yes"), n = n(), .groups = "drop")
by_arm

prop.test(x = by_arm$x, n = by_arm$n, correct = FALSE)

tab <- table(study_data$yoga, study_data$maternal_diabetes)
chisq.test(tab)$expected   # all should be >= 5
chisq.test(tab)

## Interpretation

In this simulation the true causal effect of yoga is zero, so any crude yoga, diabetes association comes from SES confounding plus random variation. We checked the confounder first: the SES-stratified summaries and faceted plots show the yoga difference shrinking inside each SES level. The crude 2×2 test then addresses only the "by chance" explanation under its assumptions; a small *p*-value cannot rule out the confounding we already saw, which is why causal language must stay cautious in an observational study.

For your one-page report, paste your summaries, a plot, and the `prop.test` p-value and 95% CI, then write a conclusion that names the plausible explanations (causal / confounding / chance) for your design.